In [2]:
# Competition-safe setup
import os
import subprocess
import sys

WHEEL_DIR_CANDIDATES = [
    "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
    os.path.join(os.getcwd(), "arc_agi_3_wheels"),
]

wheel_dir = next((path for path in WHEEL_DIR_CANDIDATES if os.path.isdir(path)), None)

if wheel_dir:
    install_cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--find-links",
        wheel_dir,
        "arc-agi",
        "arcengine",
        "python-dotenv",
    ]

    result = subprocess.run(install_cmd, check=False, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-2000:])
    if result.returncode != 0:
        print("Wheel install returned non-zero exit code; continuing with preinstalled packages.")
        if result.stderr:
            print(result.stderr[-2000:])
else:
    print("Wheel directory not found. Using preinstalled notebook packages.")

print(f"Python version: {sys.version.split()[0]}")
print("Setup cell complete.")

Looking in links: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3/arc_agi_3_wheels
Processing ./arc_agi_3_wheels/arc_agi-0.9.8-py3-none-any.whl

Wheel install returned non-zero exit code; continuing with preinstalled packages.
ERROR: Package 'arc-agi' requires a different Python: 3.9.5 not in '>=3.12'

Python version: 3.9.5
Setup cell complete.


## Shared Imports

Run this cell once before executing the cells below.

In [3]:
import csv
import glob
import json
import os
import random
import sys
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

print("Shared imports loaded.")

Shared imports loaded.


## Data and Environment Overview

In this section we:
- read all `metadata.json` files from `environment_files`
- build a table for each game (`game_id`, tags, baseline actions)
- inspect tag distribution and save an overview CSV

In [4]:
def detect_workspace_root() -> str:
    override = os.environ.get("ARC_DATA_ROOT") or os.environ.get("ARC_WORKSPACE")
    if override:
        override = os.path.abspath(os.path.expanduser(override))
        if os.path.isdir(os.path.join(override, "environment_files")):
            return override

    current = os.path.abspath(os.getcwd())
    while True:
        if os.path.isdir(os.path.join(current, "environment_files")):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent

    kaggle_root = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3"
    if os.path.isdir(os.path.join(kaggle_root, "environment_files")):
        return kaggle_root

    raise FileNotFoundError(
        "Cannot find project root with 'environment_files'. "
        "Set ARC_DATA_ROOT or ARC_WORKSPACE."
    )


def detect_output_root(data_root: str) -> str:
    candidates = [
        os.environ.get("ARC_OUTPUT_ROOT"),
        "/kaggle/working",
        os.getcwd(),
        data_root,
        "/tmp",
    ]

    for candidate in candidates:
        if not candidate:
            continue
        path = os.path.abspath(os.path.expanduser(candidate))
        if os.path.isdir(path) and os.access(path, os.W_OK):
            return path

    raise OSError(
        "Cannot find writable output directory. "
        "Set ARC_OUTPUT_ROOT to a writable path."
    )


WORKSPACE = detect_workspace_root()
OUTPUT_ROOT = detect_output_root(WORKSPACE)
os.environ["ARC_DATA_ROOT"] = WORKSPACE
os.environ["ARC_OUTPUT_ROOT"] = OUTPUT_ROOT
ENV_DIR = os.path.join(WORKSPACE, "environment_files")

metadata_paths = sorted(glob.glob(os.path.join(ENV_DIR, "*", "*", "metadata.json")))
records = []

for metadata_path in metadata_paths:
    with open(metadata_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    baseline_actions = data.get("baseline_actions", [])
    tags = data.get("tags", [])

    env_py_candidates = sorted(
        glob.glob(os.path.join(os.path.dirname(metadata_path), "*.py"))
    )
    env_py_path = (
        os.path.relpath(env_py_candidates[0], WORKSPACE) if env_py_candidates else ""
    )

    records.append(
        {
            "game_id": data.get("game_id", ""),
            "title": data.get("title", ""),
            "tags": tags,
            "num_levels": len(baseline_actions),
            "baseline_min": min(baseline_actions) if baseline_actions else None,
            "baseline_max": max(baseline_actions) if baseline_actions else None,
            "baseline_avg": (
                round(sum(baseline_actions) / len(baseline_actions), 2)
                if baseline_actions
                else None
            ),
            "metadata_path": os.path.relpath(metadata_path, WORKSPACE),
            "env_py_path": env_py_path,
        }
    )

games_df = pd.DataFrame(records).sort_values("game_id").reset_index(drop=True)
view_df = games_df.copy()
view_df["tags"] = view_df["tags"].apply(lambda x: ", ".join(x) if x else "no_tags")

all_tags = []
for tags in games_df["tags"]:
    if tags:
        all_tags.extend(tags)
    else:
        all_tags.append("no_tags")

tag_df = (
    pd.Series(all_tags, name="tag")
    .value_counts()
    .rename_axis("tag")
    .reset_index(name="count")
)

print(f"Workspace root: {WORKSPACE}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Total public games: {len(view_df)}")
print(f"Total public levels: {int(view_df['num_levels'].sum())}")
print(f"Average levels per game: {view_df['num_levels'].mean():.2f}")
print()
print("Tag distribution:")
display(tag_df)

print("\nPublic games overview:")
display(
    view_df[
        [
            "game_id",
            "title",
            "tags",
            "num_levels",
            "baseline_min",
            "baseline_max",
            "baseline_avg",
            "env_py_path",
        ]
    ]
)

overview_csv = os.path.join(OUTPUT_ROOT, "public_games_overview.csv")
view_df.to_csv(overview_csv, index=False)
print(f"\nOverview saved to {overview_csv}")

Workspace root: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3
Output root: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3
Total public games: 25
Total public levels: 183
Average levels per game: 7.32

Tag distribution:


,tag,count
0,keyboard_click,13
1,click,7
2,keyboard,4
3,no_tags,1



Public games overview:


,game_id,title,tags,num_levels,baseline_min,baseline_max,baseline_avg,env_py_path
0,ar25-0c556536,AR25,keyboard_click,8,32,233,93.50,environment_files/ar25/0c556536/ar25.py
1,bp35-0a0ad940,BP35,keyboard_click,9,21,163,72.33,environment_files/bp35/0a0ad940/bp35.py
2,cd82-fb555c5d,CD82,keyboard_click,6,8,55,28.50,environment_files/cd82/fb555c5d/cd82.py
3,cn04-2fe56bfb,CN04,keyboard_click,6,29,300,131.50,environment_files/cn04/2fe56bfb/cn04.py
4,dc22-fdcac232,DC22,keyboard_click,6,59,578,204.67,environment_files/dc22/fdcac232/dc22.py
5,ft09-0d8bbf25,FT09,no_tags,6,12,65,34.67,environment_files/ft09/0d8bbf25/ft09.py
6,g50t-5849a774,G50T,keyboard,7,54,230,125.57,environment_files/g50t/5849a774/g50t.py
7,ka59-38d34dbb,KA59,keyboard_click,7,28,326,104.29,environment_files/ka59/38d34dbb/ka59.py
8,lf52-271a04aa,LF52,click,10,32,244,133.90,environment_files/lf52/271a04aa/lf52.py
9,lp85-305b61c3,LP85,click,8,16,159,48.50,environment_files/lp85/305b61c3/lp85.py



Overview saved to /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3/public_games_overview.csv


## Experiment Infrastructure

In this section we create a minimal experiment framework:
- fixed random seed for reproducibility
- timestamped run directory
- standard files for config and metrics logging
- helper function to append metric rows

In [5]:
# Experiment setup
seed = 42
np.random.seed(seed)
random.seed(seed)


def detect_data_root() -> str:
    explicit = (
        os.environ.get("ARC_DATA_ROOT")
        or os.environ.get("ARC_WORKSPACE_ROOT")
        or os.environ.get("ARC_WORKSPACE")
    )
    if explicit:
        explicit = os.path.abspath(os.path.expanduser(explicit))
        if os.path.isdir(os.path.join(explicit, "environment_files")):
            return explicit

    if "WORKSPACE" in globals():
        candidate = os.path.abspath(str(WORKSPACE))
        if os.path.isdir(os.path.join(candidate, "environment_files")):
            return candidate

    probe = os.path.abspath(os.getcwd())
    while True:
        if os.path.isdir(os.path.join(probe, "environment_files")):
            return probe
        parent = os.path.dirname(probe)
        if parent == probe:
            break
        probe = parent

    kaggle_root = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3"
    if os.path.isdir(os.path.join(kaggle_root, "environment_files")):
        return kaggle_root

    raise FileNotFoundError(
        "Could not locate data root containing environment_files. "
        "Set ARC_DATA_ROOT or ARC_WORKSPACE_ROOT."
    )


def detect_output_root(data_root: str) -> str:
    candidates = [
        os.environ.get("ARC_OUTPUT_ROOT"),
        "/kaggle/working",
        os.getcwd(),
        data_root,
        "/tmp",
    ]

    for candidate in candidates:
        if not candidate:
            continue
        path = os.path.abspath(os.path.expanduser(candidate))
        if os.path.isdir(path) and os.access(path, os.W_OK):
            return path

    raise OSError(
        "Could not locate writable output root. "
        "Set ARC_OUTPUT_ROOT to a writable directory."
    )


def assert_runtime_budget(max_seconds: int = 5 * 3600 + 45 * 60) -> None:
    if max_seconds >= 6 * 3600:
        raise ValueError("Runtime budget must be below 6 hours for this competition.")


def log_metric(
    metrics_csv: str,
    phase: str,
    game_id: Optional[str],
    metric: str,
    value: float,
    source: str,
) -> None:
    current_ts = datetime.now(timezone.utc).isoformat()
    file_exists = os.path.exists(metrics_csv)
    with open(metrics_csv, "a", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["timestamp_utc", "phase", "game_id", "metric", "value", "source"],
        )
        if not file_exists:
            writer.writeheader()
        writer.writerow(
            {
                "timestamp_utc": current_ts,
                "phase": phase,
                "game_id": game_id,
                "metric": metric,
                "value": value,
                "source": source,
            }
        )


def setup_experiment_environment(data_root: str, output_root: str) -> Dict[str, str]:
    run_id = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S")
    run_dir = os.path.join(output_root, "runs", run_id)
    artifacts_dir = os.path.join(run_dir, "artifacts")
    logs_dir = os.path.join(run_dir, "logs")
    os.makedirs(artifacts_dir, exist_ok=True)
    os.makedirs(logs_dir, exist_ok=True)

    config_path = os.path.join(run_dir, "config.json")
    metrics_csv = os.path.join(logs_dir, "metrics.csv")

    config = {
        "run_id": run_id,
        "seed": seed,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "workspace_root": data_root,
        "data_root": data_root,
        "output_root": output_root,
        "internet_allowed": False,
        "max_runtime_seconds": 5 * 3600 + 45 * 60,
    }
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)

    return {
        "run_id": run_id,
        "run_dir": run_dir,
        "artifacts_dir": artifacts_dir,
        "logs_dir": logs_dir,
        "config_path": config_path,
        "metrics_csv": metrics_csv,
        "workspace_root": data_root,
        "data_root": data_root,
        "output_root": output_root,
    }


assert_runtime_budget()
data_root = detect_data_root()
output_root = detect_output_root(data_root)
os.environ["ARC_DATA_ROOT"] = data_root
os.environ["ARC_OUTPUT_ROOT"] = output_root
RUN_CONTEXT = setup_experiment_environment(data_root, output_root)

log_metric(
    RUN_CONTEXT["metrics_csv"],
    phase="phase3",
    game_id=None,
    metric="run_initialized",
    value=1.0,
    source="infrastructure",
)

print("Data root:", RUN_CONTEXT["data_root"])
print("Output root:", RUN_CONTEXT["output_root"])
print("Run initialized:", RUN_CONTEXT["run_id"])
print("Run directory:", RUN_CONTEXT["run_dir"])
print("Config path:", RUN_CONTEXT["config_path"])
print("Metrics CSV:", RUN_CONTEXT["metrics_csv"])

Data root: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3
Output root: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3
Run initialized: run_20260427_214109
Run directory: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3/runs/run_20260427_214109
Config path: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3/runs/run_20260427_214109/config.json
Metrics CSV: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3/runs/run_20260427_214109/logs/metrics.csv


## Baseline Agent Implementation

This section adds a lightweight baseline policy that does not require ARC runtime packages.

What it does:
- builds a baseline profile per public game from metadata
- defines a simple action policy (`RESET` + random/hybrid actions)
- runs a short dry-run simulation for a few games
- saves baseline artifacts for later comparison

In [6]:
# Load metadata prepared in Cell 6.
if "games_df" in globals():
    metadata_df = games_df.copy()
elif "overview_csv" in globals() and os.path.exists(overview_csv):
    metadata_df = pd.read_csv(overview_csv)
else:
    raise RuntimeError("Run Cell 6 first to prepare metadata.")

def normalize_tags(raw_tags) -> str:
    if isinstance(raw_tags, list):
        tags = [str(tag).strip() for tag in raw_tags if str(tag).strip()]
        return "|".join(tags) if tags else "no_tags"

    if raw_tags is None:
        return "no_tags"

    text = str(raw_tags).strip()
    if not text or text.lower() == "nan":
        return "no_tags"

    if "|" in text:
        tags = [item.strip() for item in text.split("|") if item.strip()]
        return "|".join(tags) if tags else "no_tags"

    if "," in text:
        tags = [item.strip() for item in text.split(",") if item.strip()]
        return "|".join(tags) if tags else "no_tags"

    return text


metadata_df["tags"] = metadata_df["tags"].apply(normalize_tags)

profile_columns = [
    col
    for col in ["num_levels", "baseline_min", "baseline_max", "baseline_avg"]
    if col in metadata_df.columns
]

baseline_profile = (
    metadata_df.set_index("game_id")[profile_columns].to_dict(orient="index")
    if profile_columns
    else {}
)

def baseline_choose_action(game_state: Dict, turn_idx: int, game_tags: List[str]) -> str:
    if turn_idx == 0:
        return "RESET"

    actions = game_state["available_actions"]

    if "strategy-heavy" in game_tags and "QUICK_ACTION" in actions:
        return "QUICK_ACTION"

    if random.random() < 0.15 and "RANDOM_ACTION" in actions:
        return "RANDOM_ACTION"

    return random.choice(actions)


public_game_ids = metadata_df["game_id"].dropna().astype(str).tolist()
selected_game_ids = public_game_ids[: min(5, len(public_game_ids))]

simulation_rows = []

for game_id in selected_game_ids:
    tags_value = metadata_df.loc[metadata_df["game_id"] == game_id, "tags"].iloc[0]
    tags = [] if tags_value == "no_tags" else tags_value.split("|")

    game_state = {
        "game_id": game_id,
        "score": 0,
        "available_actions": ["UP", "DOWN", "LEFT", "RIGHT", "RESET", "RANDOM_ACTION"],
    }

    for turn_idx in range(8):
        action = baseline_choose_action(game_state, turn_idx, tags)

        confidence = 0.6
        if action == "RESET":
            confidence = 0.85
        elif action == "QUICK_ACTION":
            confidence = 0.75

        simulation_rows.append(
            {
                "game_id": game_id,
                "turn": turn_idx,
                "action": action,
                "confidence": confidence,
                "tags": "|".join(tags) if tags else "no_tags",
            }
        )

        log_metric(
            RUN_CONTEXT["metrics_csv"],
            phase="phase4",
            game_id=game_id,
            metric="action_confidence",
            value=float(confidence),
            source="baseline_policy",
        )

simulation_df = pd.DataFrame(simulation_rows)
if not simulation_df.empty:
    simulation_df = simulation_df.sort_values(["game_id", "turn"]).reset_index(drop=True)

baseline_csv = os.path.join(RUN_CONTEXT["artifacts_dir"], "baseline_simulation.csv")
simulation_df.to_csv(baseline_csv, index=False)

RUN_CONTEXT["baseline_csv"] = baseline_csv

print(f"Baseline simulation rows: {len(simulation_df)}")
print(f"Saved baseline artifact: {baseline_csv}")
display(simulation_df.head(20))

Baseline simulation rows: 40
Saved baseline artifact: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3/runs/run_20260427_214109/artifacts/baseline_simulation.csv


,game_id,turn,action,confidence,tags
0,ar25-0c556536,0,RESET,0.85,keyboard_click
1,ar25-0c556536,1,UP,0.60,keyboard_click
2,ar25-0c556536,2,DOWN,0.60,keyboard_click
3,ar25-0c556536,3,RANDOM_ACTION,0.60,keyboard_click
4,ar25-0c556536,4,RANDOM_ACTION,0.60,keyboard_click
5,ar25-0c556536,5,RESET,0.85,keyboard_click
6,ar25-0c556536,6,RANDOM_ACTION,0.60,keyboard_click
7,ar25-0c556536,7,UP,0.60,keyboard_click
8,bp35-0a0ad940,0,RESET,0.85,keyboard_click
9,bp35-0a0ad940,1,RANDOM_ACTION,0.60,keyboard_click


## Offline Generative Policy

This section adds a local, internet-free generative policy.

What it does:
- builds compact textual prompts from game state and history
- generates candidate commands using stochastic templates
- validates commands against the allowed action set
- logs prompts, actions, and confidence values as artifacts

In [7]:
def build_prompt(game_state: Dict, history_actions: List[str], memory_vector: Dict[str, float]) -> str:
    lines = [
        f"Game: {game_state['game_id']}",
        f"Turn: {game_state['turn_index']}",
        f"Score: {game_state.get('score', 0)}",
        f"Available actions: {', '.join(game_state['available_actions'])}",
        f"Memory avg reward: {memory_vector.get('avg_reward', 0.0):.3f}",
        f"Memory reset success: {memory_vector.get('reset_success', 0.0):.3f}",
    ]

    if history_actions:
        lines.append("Recent history: " + " -> ".join(history_actions[-5:]))
    else:
        lines.append("Recent history: none")

    lines.append("Produce one valid action and confidence in [0,1].")
    return "\n".join(lines)


def validate_action(action: str, available_actions: List[str]) -> str:
    if action in available_actions:
        return action
    return random.choice(available_actions)


def generate_offline_action(
    game_state: Dict,
    history_actions: List[str],
    memory_vector: Dict[str, float],
) -> Tuple[str, float, str]:
    prompt = build_prompt(game_state, history_actions, memory_vector)

    template_pool = [
        "UP",
        "DOWN",
        "LEFT",
        "RIGHT",
        "RESET",
        "RANDOM_ACTION",
    ]

    base = random.choice(template_pool)
    suffix = random.choice(["", " cautiously", " aggressively", " with minimal risk"])
    candidate = f"{base}{suffix}"

    action = validate_action(candidate, game_state["available_actions"])
    confidence = float(np.clip(np.random.normal(loc=0.68, scale=0.14), 0.05, 0.98))

    return action, confidence, prompt


def generate_action(
    game_state: Dict,
    history_actions: List[str],
    memory_vector: Dict[str, float],
) -> Tuple[str, float, str]:
    return generate_offline_action(game_state, history_actions, memory_vector)

selected_ids = simulation_df["game_id"].dropna().unique().tolist()[: min(5, len(simulation_df))]

prompt_records = []
gen_rows = []

for game_id in selected_ids:
    tags = metadata_df.loc[metadata_df["game_id"] == game_id, "tags"].iloc[0].split("|")

    memory_vector = {
        "avg_reward": float(np.random.uniform(-0.2, 0.6)),
        "reset_success": float(np.random.uniform(0.2, 0.9)),
    }

    history_actions = []

    for turn_idx in range(8):
        game_state = {
            "game_id": game_id,
            "turn_index": turn_idx,
            "score": int(np.random.randint(0, 100)),
            "available_actions": ["UP", "DOWN", "LEFT", "RIGHT", "RESET", "RANDOM_ACTION"],
        }

        action, confidence, prompt = generate_action(game_state, history_actions, memory_vector)
        history_actions.append(action)

        prompt_records.append(
            {
                "game_id": game_id,
                "turn": turn_idx,
                "prompt": prompt,
                "action": action,
                "confidence": confidence,
            }
        )

        gen_rows.append(
            {
                "game_id": game_id,
                "turn": turn_idx,
                "action": action,
                "confidence": confidence,
                "tags": "|".join(tags),
                "policy": "offline_generative",
            }
        )

        log_metric(
            RUN_CONTEXT["metrics_csv"],
            phase="phase5",
            game_id=game_id,
            metric="gen_confidence",
            value=float(confidence),
            source="offline_generator",
        )

gen_df = pd.DataFrame(gen_rows).sort_values(["game_id", "turn"]).reset_index(drop=True)

gen_actions_csv = os.path.join(RUN_CONTEXT["artifacts_dir"], "generative_actions.csv")
gen_prompts_jsonl = os.path.join(RUN_CONTEXT["artifacts_dir"], "generative_prompts.jsonl")

gen_df.to_csv(gen_actions_csv, index=False)

with open(gen_prompts_jsonl, "w") as f:
    for row in prompt_records:
        f.write(json.dumps(row) + "\n")

RUN_CONTEXT["gen_actions_csv"] = gen_actions_csv
RUN_CONTEXT["gen_prompts_jsonl"] = gen_prompts_jsonl

print(f"Generated rows: {len(gen_df)}")
print(f"Saved actions: {gen_actions_csv}")
print(f"Saved prompts: {gen_prompts_jsonl}")
display(gen_df.head(20))

Generated rows: 40
Saved actions: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3/runs/run_20260427_214109/artifacts/generative_actions.csv
Saved prompts: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3/runs/run_20260427_214109/artifacts/generative_prompts.jsonl


,game_id,turn,action,confidence,tags,policy
0,ar25-0c556536,0,RANDOM_ACTION,0.524337,keyboard_click,offline_generative
1,ar25-0c556536,1,RANDOM_ACTION,0.724646,keyboard_click,offline_generative
2,ar25-0c556536,2,RESET,0.901090,keyboard_click,offline_generative
3,ar25-0c556536,3,DOWN,0.787441,keyboard_click,offline_generative
4,ar25-0c556536,4,LEFT,0.598677,keyboard_click,offline_generative
5,ar25-0c556536,5,RANDOM_ACTION,0.606476,keyboard_click,offline_generative
6,ar25-0c556536,6,LEFT,0.713875,keyboard_click,offline_generative
7,ar25-0c556536,7,DOWN,0.412141,keyboard_click,offline_generative
8,bp35-0a0ad940,0,LEFT,0.644854,keyboard_click,offline_generative
9,bp35-0a0ad940,1,RESET,0.657059,keyboard_click,offline_generative


## Memory and Reflection Layer

This section introduces lightweight episodic memory:
- keeps recent state-action records in memory
- infers a weak outcome signal from score deltas
- updates per-tag and per-action priors
- stores memory snapshots and reflection logs for analysis

In [8]:
def init_game_memory(tags: List[str]) -> Dict:
    return {
        "tag_stats": {tag: {"reward_sum": 0.0, "count": 0} for tag in tags},
        "action_stats": {},
        "short_history": [],
    }


def infer_outcome_signal(prev_score: float, next_score: float) -> float:
    delta = next_score - prev_score
    return float(np.tanh(delta / 20.0))


def apply_reflection(memory: Dict, tags: List[str], action: str, reward_signal: float, turn_idx: int) -> Dict:
    for tag in tags:
        memory["tag_stats"].setdefault(tag, {"reward_sum": 0.0, "count": 0})
        memory["tag_stats"][tag]["reward_sum"] += reward_signal
        memory["tag_stats"][tag]["count"] += 1

    memory["action_stats"].setdefault(action, {"reward_sum": 0.0, "count": 0})
    memory["action_stats"][action]["reward_sum"] += reward_signal
    memory["action_stats"][action]["count"] += 1

    memory["short_history"].append(
        {
            "turn": turn_idx,
            "action": action,
            "reward_signal": reward_signal,
        }
    )
    memory["short_history"] = memory["short_history"][-10:]

    action_count = memory["action_stats"][action]["count"]
    action_reward_sum = memory["action_stats"][action]["reward_sum"]

    return {
        "action_avg_reward": action_reward_sum / max(action_count, 1),
        "history_size": len(memory["short_history"]),
    }


if "gen_df" not in globals() or gen_df.empty:
    raise RuntimeError("Generative results are not available. Execute the previous policy cell first.")

reflection_rows = []

for game_id, group in gen_df.groupby("game_id"):
    tags = metadata_df.loc[metadata_df["game_id"] == game_id, "tags"].iloc[0].split("|")
    memory = init_game_memory(tags)

    prev_score = 0.0
    ordered_group = group.sort_values("turn")

    for _, action_row in ordered_group.iterrows():
        action = action_row["action"]
        next_score = prev_score + np.random.normal(loc=2.0, scale=5.0)
        reward_signal = infer_outcome_signal(prev_score, next_score)

        summary = apply_reflection(
            memory=memory,
            tags=tags,
            action=action,
            reward_signal=reward_signal,
            turn_idx=int(action_row["turn"]),
        )

        reflection_rows.append(
            {
                "game_id": game_id,
                "turn": int(action_row["turn"]),
                "action": action,
                "reward_signal": reward_signal,
                "action_avg_reward": summary["action_avg_reward"],
                "history_size": summary["history_size"],
            }
        )

        log_metric(
            RUN_CONTEXT["metrics_csv"],
            phase="phase6",
            game_id=game_id,
            metric="reward_signal",
            value=float(reward_signal),
            source="memory_reflection",
        )

        prev_score = next_score

reflection_df = pd.DataFrame(reflection_rows).sort_values(["game_id", "turn"]).reset_index(drop=True)

memory_reflection_csv = os.path.join(RUN_CONTEXT["artifacts_dir"], "memory_reflection.csv")
reflection_df.to_csv(memory_reflection_csv, index=False)

RUN_CONTEXT["memory_reflection_csv"] = memory_reflection_csv

print(f"Reflection rows: {len(reflection_df)}")
print(f"Saved reflection artifact: {memory_reflection_csv}")
display(reflection_df.head(20))

Reflection rows: 40
Saved reflection artifact: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3/runs/run_20260427_214109/artifacts/memory_reflection.csv


,game_id,turn,action,reward_signal,action_avg_reward,history_size
0,ar25-0c556536,0,RANDOM_ACTION,0.736631,0.736631,1
1,ar25-0c556536,1,RANDOM_ACTION,-0.129369,0.303631,2
2,ar25-0c556536,2,RESET,0.000340,0.000340,3
3,ar25-0c556536,3,DOWN,0.084581,0.084581,4
4,ar25-0c556536,4,LEFT,-0.249320,-0.249320,5
5,ar25-0c556536,5,RANDOM_ACTION,0.345763,0.317675,6
6,ar25-0c556536,6,LEFT,0.314817,0.032748,7
7,ar25-0c556536,7,DOWN,0.104369,0.094475,8
8,bp35-0a0ad940,0,LEFT,-0.033591,-0.033591,1
9,bp35-0a0ad940,1,RESET,-0.267123,-0.267123,2


## Evaluation Summary

This section computes aggregate diagnostics for baseline, generative, and memory layers.

It saves:
- compact summary table
- per-game summary table
- both files under the current run artifacts directory

In [9]:
required_frames = {
    "simulation_df": "baseline output",
    "gen_df": "generative output",
    "reflection_df": "memory output",
}
missing_frames = [name for name in required_frames if name not in globals()]
if missing_frames:
    raise RuntimeError(
        "Missing data frames: "
        + ", ".join(missing_frames)
        + ". Run Cells 10, 12, and 14 first."
    )

allowed_actions = {"UP", "DOWN", "LEFT", "RIGHT", "RESET", "RANDOM_ACTION", "QUICK_ACTION"}

baseline_eval = {
    "phase": "baseline",
    "rows": int(len(simulation_df)),
    "games": int(simulation_df["game_id"].nunique()),
    "mean_confidence": float(simulation_df["confidence"].mean()),
    "reset_share": float((simulation_df["action"] == "RESET").mean()),
    "invalid_action_share": float((~simulation_df["action"].isin(allowed_actions)).mean()),
    "mean_reward_signal": float("nan"),
}

gen_eval = {
    "phase": "generative",
    "rows": int(len(gen_df)),
    "games": int(gen_df["game_id"].nunique()),
    "mean_confidence": float(gen_df["confidence"].mean()),
    "reset_share": float((gen_df["action"] == "RESET").mean()),
    "invalid_action_share": float((~gen_df["action"].isin(allowed_actions)).mean()),
    "mean_reward_signal": float("nan"),
}

memory_eval = {
    "phase": "memory",
    "rows": int(len(reflection_df)),
    "games": int(reflection_df["game_id"].nunique()),
    "mean_confidence": float("nan"),
    "reset_share": float((reflection_df["action"] == "RESET").mean()),
    "invalid_action_share": float((~reflection_df["action"].isin(allowed_actions)).mean()),
    "mean_reward_signal": float(reflection_df["reward_signal"].mean()),
}

evaluation_summary_df = pd.DataFrame([baseline_eval, gen_eval, memory_eval])

game_baseline_df = (
    simulation_df.groupby("game_id", as_index=False)
    .agg(
        baseline_turns=("turn", "count"),
        baseline_confidence_mean=("confidence", "mean"),
        baseline_reset_share=("action", lambda s: float((s == "RESET").mean())),
    )
)

game_gen_df = (
    gen_df.groupby("game_id", as_index=False)
    .agg(
        gen_turns=("turn", "count"),
        gen_confidence_mean=("confidence", "mean"),
        gen_reset_share=("action", lambda s: float((s == "RESET").mean())),
    )
)

game_memory_df = (
    reflection_df.groupby("game_id", as_index=False)
    .agg(
        memory_turns=("turn", "count"),
        reward_signal_mean=("reward_signal", "mean"),
        reward_signal_std=("reward_signal", "std"),
    )
)

evaluation_by_game_df = (
    game_baseline_df
    .merge(game_gen_df, on="game_id", how="outer")
    .merge(game_memory_df, on="game_id", how="outer")
    .sort_values("game_id")
    .reset_index(drop=True)
)

evaluation_summary_csv = os.path.join(RUN_CONTEXT["artifacts_dir"], "evaluation_summary.csv")
evaluation_by_game_csv = os.path.join(RUN_CONTEXT["artifacts_dir"], "evaluation_by_game.csv")

evaluation_summary_df.to_csv(evaluation_summary_csv, index=False)
evaluation_by_game_df.to_csv(evaluation_by_game_csv, index=False)

RUN_CONTEXT["evaluation_summary_csv"] = evaluation_summary_csv
RUN_CONTEXT["evaluation_by_game_csv"] = evaluation_by_game_csv

log_metric(
    RUN_CONTEXT["metrics_csv"],
    phase="phase7",
    game_id=None,
    metric="gen_invalid_action_share",
    value=float(gen_eval["invalid_action_share"]),
    source="evaluation",
)

print("Evaluation summary saved:", evaluation_summary_csv)
print("Per-game summary saved:", evaluation_by_game_csv)
display(evaluation_summary_df)
display(evaluation_by_game_df.head(20))

Evaluation summary saved: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3/runs/run_20260427_214109/artifacts/evaluation_summary.csv
Per-game summary saved: /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3/runs/run_20260427_214109/artifacts/evaluation_by_game.csv


,phase,rows,games,mean_confidence,reset_share,invalid_action_share,mean_reward_signal
0,baseline,40,5,0.662500,0.250,0.0,NaN
1,generative,40,5,0.665022,0.225,0.0,NaN
2,memory,40,5,NaN,0.225,0.0,0.07177


,game_id,baseline_turns,baseline_confidence_mean,baseline_reset_share,gen_turns,gen_confidence_mean,gen_reset_share,memory_turns,reward_signal_mean,reward_signal_std
0,ar25-0c556536,8,0.66250,0.250,8,0.658585,0.125,8,0.150977,0.310732
1,bp35-0a0ad940,8,0.69375,0.375,8,0.712206,0.250,8,0.064182,0.212273
2,cd82-fb555c5d,8,0.66250,0.250,8,0.545222,0.250,8,0.153393,0.138149
3,cn04-2fe56bfb,8,0.66250,0.250,8,0.737809,0.125,8,0.049173,0.262148
4,dc22-fdcac232,8,0.63125,0.125,8,0.671288,0.375,8,-0.058875,0.265581


## Submission Packaging

This section assembles a competition-style output table and validates mandatory columns.

Behavior:
- uses generative actions by default
- applies baseline fallback when an action is invalid
- on Kaggle rerun, copies `ARC-AGI-3-Agents` to writable storage and runs an adaptive non-LLM agent by default
- supports manual agent selection via `ARC_AGENT_NAME` (for example: `adaptive` or `random`)
- falls back to `random` if the selected agent fails at runtime
- outside Kaggle rerun, writes a local debug submission artifact (parquet when available, otherwise CSV)

In [10]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def write_local_debug_submission() -> str:
    import numpy as np
    import pandas as pd

    if "gen_df" in globals() and isinstance(gen_df, pd.DataFrame) and not gen_df.empty:
        last_turn_by_game = (
            gen_df.groupby("game_id", as_index=False)["turn"]
            .max()
            .set_index("game_id")["turn"]
            .to_dict()
        )

        rows = []
        for row in gen_df.sort_values(["game_id", "turn"]).itertuples(index=False):
            game_id = str(row.game_id)
            turn = int(row.turn)
            confidence = float(getattr(row, "confidence", 0.5))
            rows.append(
                {
                    "row_id": f"{game_id}_{turn}",
                    "game_id": game_id,
                    "end_of_game": bool(turn == int(last_turn_by_game[game_id])),
                    "score": int(np.clip(round(confidence * 100.0), 0, 100)),
                }
            )

        submission_df = pd.DataFrame(
            rows, columns=["row_id", "game_id", "end_of_game", "score"]
        )
    else:
        submission_df = pd.DataFrame(
            data=[["1_0", "1", True, 1]],
            columns=["row_id", "game_id", "end_of_game", "score"],
        )

    target_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
    target_path = target_dir / "submission.parquet"
    try:
        submission_df.to_parquet(target_path, index=False)
    except Exception:
        target_path = target_dir / "submission.csv"
        submission_df.to_csv(target_path, index=False)
    return str(target_path)


is_rerun = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

if is_rerun:
    print("Competition rerun detected: running ARC-AGI-3-Agents via gateway.")

    subprocess.run(
        [
            "curl",
            "--fail",
            "--retry",
            "999",
            "--retry-all-errors",
            "--retry-delay",
            "5",
            "--retry-max-time",
            "600",
            "http://gateway:8001/api/games",
        ],
        check=True,
    )

    input_root = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3")
    source_repo = input_root / "ARC-AGI-3-Agents"
    target_repo = Path("/kaggle/working/ARC-AGI-3-Agents")

    if not source_repo.is_dir():
        raise FileNotFoundError(f"Missing competition repo at {source_repo}")

    if target_repo.exists():
        shutil.rmtree(target_repo)
    shutil.copytree(source_repo, target_repo)

    adaptive_agent_py = target_repo / "agents" / "templates" / "adaptive_agent.py"
    adaptive_agent_py.write_text(
        '''from __future__ import annotations
import math
import random
import time
from collections import defaultdict
from typing import Any, Dict, List, Optional, Set, Tuple

from arcengine import FrameData, GameAction, GameState

from ..agent import Agent


class AdaptiveHeuristic(Agent):
    """Reward and novelty driven non-LLM policy."""

    MAX_ACTIONS = 220
    UCB_C = 1.4
    CLICK_STRIDE = 8

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        seed = int(time.time() * 1_000_000) + hash(self.game_id) % 1_000_000
        random.seed(seed)
        self.q: Dict[str, float] = defaultdict(float)
        self.n: Dict[str, int] = defaultdict(int)
        self.total_steps = 0
        self.prev_action_key: Optional[str] = None
        self.seen_hashes: Set[int] = set()

    @property
    def name(self) -> str:
        return f"{super().name}.{self.MAX_ACTIONS}"

    def is_done(self, frames: List[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def _grid_hash(self, frame: FrameData) -> int:
        if not frame.frame:
            return 0
        grid = frame.frame[-1]
        return hash(tuple(tuple(int(v) for v in row) for row in grid))

    def _update_reward(self, frames: List[FrameData]) -> None:
        if len(frames) < 2 or self.prev_action_key is None:
            return

        previous = frames[-2]
        current = frames[-1]

        reward = 0.0
        delta_levels = int(current.levels_completed) - int(previous.levels_completed)
        reward += float(delta_levels) * 50.0

        current_hash = self._grid_hash(current)
        if current_hash not in self.seen_hashes:
            self.seen_hashes.add(current_hash)
            reward += 1.0
        else:
            reward -= 0.25

        if previous.frame and current.frame and previous.frame[-1] == current.frame[-1]:
            reward -= 0.75

        if current.state is GameState.GAME_OVER:
            reward -= 2.0

        key = self.prev_action_key
        old_q = float(self.q[key])
        old_n = int(self.n[key])
        new_n = old_n + 1
        self.n[key] = new_n
        self.q[key] = old_q + (reward - old_q) / float(new_n)

    def _candidate_click_points(self, latest_frame: FrameData) -> List[Tuple[int, int]]:
        points: List[Tuple[int, int]] = []

        if latest_frame.frame:
            grid = latest_frame.frame[-1]
            height = len(grid)
            width = len(grid[0]) if height else 0

            if height and width:
                y_limit = min(height, 64)
                x_limit = min(width, 64)

                for y in range(0, y_limit, self.CLICK_STRIDE):
                    row = grid[y]
                    for x in range(0, x_limit, self.CLICK_STRIDE):
                        value = int(row[x])
                        if value not in (0, 1, 2, 3, 4, 5):
                            points.append((x, y))

                centers = [
                    (width // 2, height // 2),
                    (width // 3, height // 3),
                    (2 * width // 3, 2 * height // 3),
                    (width // 2, height // 3),
                    (width // 3, height // 2),
                ]
                for x, y in centers:
                    points.append((max(0, min(63, int(x))), max(0, min(63, int(y)))))

        if not points:
            points = [
                (32, 32),
                (16, 16),
                (48, 48),
                (16, 48),
                (48, 16),
                (32, 16),
                (16, 32),
                (48, 32),
                (32, 48),
            ]

        deduped: List[Tuple[int, int]] = []
        seen: Set[Tuple[int, int]] = set()
        for point in points:
            if point not in seen:
                seen.add(point)
                deduped.append(point)

        return deduped

    def _build_candidates(
        self, latest_frame: FrameData
    ) -> List[Tuple[GameAction, str, Optional[Dict[str, int]]]]:
        if latest_frame.available_actions:
            available_actions = list(latest_frame.available_actions)
        else:
            available_actions = list(GameAction)

        non_reset_actions = [
            action for action in available_actions if action is not GameAction.RESET
        ]
        if not non_reset_actions:
            non_reset_actions = [
                GameAction.ACTION1,
                GameAction.ACTION2,
                GameAction.ACTION3,
                GameAction.ACTION4,
                GameAction.ACTION5,
            ]

        click_points = self._candidate_click_points(latest_frame)
        candidates: List[Tuple[GameAction, str, Optional[Dict[str, int]]]] = []

        for action in non_reset_actions:
            if action.is_complex():
                for x, y in click_points[:12]:
                    payload = {"x": int(x), "y": int(y)}
                    key = f"{action.name}:{x}:{y}"
                    candidates.append((action, key, payload))
            else:
                candidates.append((action, action.name, None))

        return candidates

    def choose_action(
        self, frames: List[FrameData], latest_frame: FrameData
    ) -> GameAction:
        self._update_reward(frames)

        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            self.prev_action_key = "RESET"
            reset_action = GameAction.RESET
            reset_action.reasoning = "Reset to continue exploration."
            return reset_action

        candidates = self._build_candidates(latest_frame)
        self.total_steps += 1

        selected: Optional[Tuple[GameAction, str, Optional[Dict[str, int]]]] = None
        selected_score = float("-inf")

        for action, key, payload in candidates:
            count = int(self.n.get(key, 0))
            value = float(self.q.get(key, 0.0))
            if count == 0:
                score = 1_000_000.0
            else:
                score = value + self.UCB_C * math.sqrt(
                    math.log(float(self.total_steps + 1)) / float(count)
                )

            if score > selected_score:
                selected_score = score
                selected = (action, key, payload)

        if selected is None:
            selected_action = random.choice(
                [action for action in GameAction if action is not GameAction.RESET]
            )
            selected_key = selected_action.name
            selected_payload: Optional[Dict[str, int]] = None
        else:
            selected_action, selected_key, selected_payload = selected

        if selected_payload is not None:
            selected_action.set_data(selected_payload)
            selected_action.reasoning = {
                "policy": "adaptive_ucb",
                "score": round(float(selected_score), 4),
                "x": selected_payload["x"],
                "y": selected_payload["y"],
            }
        else:
            selected_action.reasoning = f"adaptive_ucb score={selected_score:.4f}"

        self.prev_action_key = selected_key
        return selected_action
''',
        encoding="utf-8",
    )

    init_py = target_repo / "agents" / "__init__.py"
    init_py.write_text(
        "from typing import Type\n"
        "from dotenv import load_dotenv\n"
        "from .agent import Agent, Playback\n"
        "from .swarm import Swarm\n"
        "from .templates.random_agent import Random\n"
        "from .templates.adaptive_agent import AdaptiveHeuristic\n"
        "\n"
        "load_dotenv()\n"
        "\n"
        "AVAILABLE_AGENTS: dict[str, Type[Agent]] = {\n"
        '    "random": Random,\n'
        '    "adaptive": AdaptiveHeuristic,\n'
        '    "adaptiveheuristic": AdaptiveHeuristic,\n'
        "}\n",
        encoding="utf-8",
    )

    env_file = target_repo / ".env"
    env_file.write_text(
        "SCHEME=http\n"
        "HOST=gateway\n"
        "PORT=8001\n"
        "ARC_API_KEY=test-key-123\n"
        "ARC_BASE_URL=http://gateway:8001/\n"
        "OPERATION_MODE=online\n"
        "ENVIRONMENTS_DIR=\n"
        "RECORDINGS_DIR=/kaggle/working/server_recording\n",
        encoding="utf-8",
    )

    run_env = os.environ.copy()
    run_env["MPLBACKEND"] = "agg"
    agent_name = os.getenv("ARC_AGENT_NAME", "adaptive").strip().lower() or "adaptive"
    print(f"Requested agent: {agent_name}")

    try:
        subprocess.run(
            [sys.executable, "main.py", "--agent", agent_name],
            cwd=str(target_repo),
            env=run_env,
            check=True,
        )
    except subprocess.CalledProcessError:
        if agent_name != "random":
            print("Selected agent failed. Falling back to random.")
            subprocess.run(
                [sys.executable, "main.py", "--agent", "random"],
                cwd=str(target_repo),
                env=run_env,
                check=True,
            )
        else:
            raise

    submission_target = Path("/kaggle/working/submission.parquet")
    if not submission_target.is_file():
        raise FileNotFoundError(
            f"Expected {submission_target} after agent run, but it was not created."
        )
    print(f"Submission created at {submission_target}")
else:
    debug_path = write_local_debug_submission()
    print(f"Non-rerun mode: wrote local debug submission to {debug_path}")

Non-rerun mode: wrote local debug submission to /home/anatolii/Education/University/AI/ARC_Prize_2026-ARC-AGI-3/submission.csv
